# LANGCHAIN으로 RAG 시작하기


Gemini 대신 **OpenAI API** 를 사용


### Step 0 : 설치와 준비

Colab에서 필요한 라이브러리를 설치하고, OpenAI API 키를 **Colab 환경변수(userdata)** 로 불러오기


In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-openai langchain-chroma chromadb langchain-classic langgraph langgraph-prebuilt google-adk faiss-cpu > /dev/null 2>&1

In [ ]:
# InMemoryVectorStore는 추가 벡터 DB 패키지가 거의 필요 없어 Colab 충돌을 크게 줄여줌
# !pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-openai langchain-chroma chromadb langchain-classic langgraph langgraph-prebuilt google-adk faiss-cpu > /dev/null 2>&1

!pip install -q -U \
  "langchain==0.3.27" \
  "langchain-community==0.3.27" \
  "langchain-openai==0.3.28" \
  "langchain-text-splitters==0.3.9" \
  "langchain-core==0.3.83" \
  "openai>=1.40,<2" \
  "pypdf>=5,<6" \
  "tiktoken>=0.7,<1" \
  "beautifulsoup4>=4.12,<5" \
  "lxml>=5,<6"

# 설치 후 import 에러가 남아 있으면 런타임을 다시 시작한 뒤 위에서부터 다시 실행

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00


In [ ]:
# Colab userdata에서 API 키를 읽어 환경변수로 등록
# WebBaseLoader 경고를 줄이기 위해 USER_AGENT도 함께 지정
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get("OpenAIapi")
os.environ['USER_AGENT'] = 'colab-langchain-rag-practice/1.0'


In [ ]:
# OpenAI 기반 LLM을 준비
# 이후 Retrieval 단계에서 찾아온 문맥을 바탕으로 답변을 생성하는 역할
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
# RetrievalQA는 템플릿 흐름 유지를 위해 호환 버전에서 사용
from langchain.chains import RetrievalQA


### Step 1 : Document Loaders 사용해보기

Document Loader는 PDF, CSV, 웹페이지 같은 원본 데이터를 **LangChain의 Document 객체** 로 바꾸는 단계


#### PDFLoader 사용

실습용 PDF 파일 `Demian.pdf` 를 Colab 세션에 업로드한 뒤 `/content/Demian.pdf` 경로에서 불러오기
PyPDFLoader는 보통 **페이지 단위 Document** 로 문서를 나눠줍니다.


In [ ]:
# 최신 문서 기준으로는 load() 후 splitter를 별도로 적용하는 방식이 더 안정적
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load()

In [ ]:
# 첫 번째 페이지 Document 구조를 확인
pages[0]


Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [ ]:
# Document 전체 대신 실제 본문 텍스트만 확인
print(pages[0].page_content[:1000])


DEMIAN 
• 
Downloaded from https://www.holybooks.com


#### CSVLoader

CSV는 행(row) 단위의 구조화 데이터
CSVLoader를 사용하면 각 행을 하나의 Document처럼 다뤄, 이후 임베딩과 검색에 활용할 수 있음


In [ ]:
# CSV 파일의 각 행을 Document로 변환
loader = CSVLoader("/content/titanic.csv")
data = loader.load()


In [ ]:
# 앞쪽 일부 행이 어떻게 Document로 바뀌었는지 확인
data[:3]


[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스 로더

WebBaseLoader는 웹페이지의 텍스트를 직접 읽어와 Document 객체로 변환
아래 예시는 외부 URL을 사용하는 예제이므로, 네트워크 환경에 따라 실행 시간이 달라질 수 있음


In [ ]:
# 웹페이지 본문을 Document 형태로 로드
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()


In [ ]:
# 웹 문서의 앞부분 텍스트를 확인
print(documents[0].page_content)






































모두의연구소 ‘AI학교 아이펠’, ICLR 2024 워크숍 혁신 기술 논문 채택 < 일반 < 기업 < 기사본문 - IT조선










 






















































주요서비스 바로가기
본문 바로가기
매체정보 바로가기
로그인 바로가기
기사검색 바로가기
전체서비스 바로가기




























 






전체메뉴



기사검색








기사검색


검색

닫기














기사검색









기사검색


검색

닫기











로그인




facebook

post
youtube



UPDATED. 2026-03-17 07:00 (화) 




기사검색









기업

일반
모바일·가전
방송·통신
반도체·디스플레이
SW·보안
중공업·에너지
소부장·스타트업
유통·쇼핑
프롭테크·부동산



모빌리티

자동차·모빌리티
로봇·드론·항공
방산



게임·콘텐츠

게임·인터넷
메타버스·VR
키덜트
미디어·엔터



과학·헬스

과학·우주
의학·정책
제약·바이오



파이낸스

금융
증권
핀테크·블록체인



칼럼·인터뷰

칼럼
기고
인터뷰



알림

알립니다
인사
부음
새로나왔어요



컴퓨팅·AI

일반
코딩
에듀테크
테크리포트



GLOBAL
기획·연재
속보
백과사전











속보




기업


일반


모바일·가전


방송·통신


반도체·디스플레이


SW·보안


중공업·에너지


소부장·스타트업


유통·쇼핑


프롭테크·부동산




모빌리티


자동차·모빌리티


로봇·드론·항공


방산




게임·콘텐츠


게임·인터넷


메타버스·VR


키덜트


미디어·엔터




과학·헬스


과학·우주


의학·정책


제약·바이오




파이낸스


금융


증권


핀테크·블록체인




칼럼·인터뷰


칼럼


기고


인터뷰



### Step 2 : TextSplitters 사용해보기

LLM은 긴 문서를 한 번에 모두 처리하기 어렵기 때문에, 텍스트를 **작은 Chunk** 로 나누는 과정이 필요  
이 Chunk가 RAG에서 검색되는 기본 단위


In [ ]:
# 긴 텍스트 파일을 읽어와 chunk 분할 실습에 사용
with open("/content/state_of_the_union.txt") as f:
    text = f.read()


In [ ]:
# CharacterTextSplitter는 단순한 구분자 기준으로 텍스트를 나눔
# chunk_size는 한 조각의 최대 길이, chunk_overlap은 앞뒤 chunk가 겹치는 길이
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_text(text)


In [ ]:
# 첫 번째 chunk 내용을 확인
print(chunks[0])


Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


In [ ]:
# 각 chunk의 문자 길이를 확인
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)


[939, 916, 994, 874, 906, 923, 951, 967, 952, 930, 962, 956, 973, 974, 858, 973, 993, 958, 978, 956, 842, 889, 942, 975, 974, 892, 977, 958, 951, 924, 975, 966, 998, 965, 845, 832, 931, 975, 954, 771, 838, 932, 957, 966, 930, 902, 911, 939, 581]


### 토큰 단위로 텍스트 분할해보기

문자 수와 실제 토큰 수는 다를 수 있으므로, 토큰 기준 길이를 확인해보는 것이 중요
특히 OpenAI 모델은 토큰 단위로 입력 길이를 계산


In [ ]:
# OpenAI 계열 모델에서 자주 쓰는 cl100k_base 토크나이저를 사용
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)


In [ ]:
# 같은 chunk라도 문자 수와 토큰 수가 다를 수 있음을 확인
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)


[939, 916, 994, 874, 906, 923, 951, 967, 952, 930, 962, 956, 973, 974, 858, 973, 993, 958, 978, 956, 842, 889, 942, 975, 974, 892, 977, 958, 951, 924, 975, 966, 998, 965, 845, 832, 931, 975, 954, 771, 838, 932, 957, 966, 930, 902, 911, 939, 581]
[197, 183, 204, 173, 188, 185, 195, 203, 197, 205, 214, 214, 214, 208, 181, 212, 218, 215, 211, 211, 197, 189, 214, 213, 193, 182, 215, 211, 202, 204, 214, 210, 236, 206, 166, 176, 200, 195, 190, 156, 173, 193, 188, 214, 191, 196, 197, 209, 142]


In [ ]:
# 실제 RAG에서는 구조를 좀 더 잘 보존하는 RecursiveCharacterTextSplitter를 많이 사용
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len
)

state_chunks = text_splitter.split_text(text)
len(state_chunks)

19

### Step 3 : TextEmbedding 사용해보기

Embedding은 텍스트를 숫자 벡터로 바꾸는 단계
RAG에서는 질문과 문서를 같은 벡터 공간에 올려, 의미적으로 비슷한 내용을 찾는 데 사용

In [ ]:
# OpenAI 임베딩 모델을 준비
# Retrieval 성능에 직접 영향을 주는 핵심 컴포넌트
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
# 여러 문장을 임베딩하여 벡터로 변환
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)


In [ ]:
# 한 문장의 임베딩 벡터 일부와 차원 수를 확인
print(embeddings[1][:10])
print(len(embeddings[1]))


[0.006435394287109375, -0.0208587646484375, -0.0216064453125, 0.0235595703125, -0.0428466796875, -0.004608154296875, 0.03973388671875, 0.05267333984375, -0.0018892288208007812, -0.0237579345703125]
1536


In [ ]:
# 코사인 유사도로 query와 각 문장의 의미적 유사도를 비교
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

query = ["this is red fruit"]
e_query = embedding_model.embed_documents(query)

print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))


0.7478342692665315
0.4898954244712594
0.40836637441688617


### Step 4 : VectorStore 사용해보기

VectorStore는 문서를 임베딩한 뒤 저장하고, 비슷한 벡터를 빠르게 찾는 저장소

지금은 Colab에서 추가 설치 충돌이 적은 **InMemoryVectorStore** 를 사용


In [ ]:
# PDF 문서를 페이지 단위 Document로 로드
# 최신 문서 기준으로는 load() 후 splitter를 별도로 적용하는 방식이 더 안정적
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    length_function=tiktoken_len
)

docs = text_splitter.split_documents(pages)

In [ ]:
# 분할된 문서를 메모리 기반 벡터스토어에 저장
# 학습용 실습에서는 별도 DB 없이도 유사도 검색 흐름을 확인
db = InMemoryVectorStore.from_documents(documents=docs, embedding=embedding_model)


In [ ]:
# 질문과 유사한 문서를 먼저 검색
query = "how Demian look like?"
docs = db.similarity_search(query, k=4)

print(docs[0].page_content)


DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

### Step 5 : Retriever 사용해보기

Retriever는 사용자의 질문과 가장 관련 있는 Chunk를 찾아 LLM에 넘겨주는 역할

VectorStore와 LLM을 결합해 실제 RAG 질의응답 체인을 만들어보기


In [ ]:
qa = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=db.as_retriever(
        search_type="mmr",  # MMR은 관련성과 다양성을 함께 고려
        search_kwargs={"k": 3, "fetch_k": 10}  # 최종 3개를 고르기 전에 10개 후보를 먼저 봄
    ),
    return_source_documents=True
)


In [ ]:
# RAG 체인으로 질문해보고, 검색된 근거를 바탕으로 답변을 생성
query = "how demian looks like"
result = qa.invoke({"query": query})

result


{'query': 'how demian looks like',
 'result': 'Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet, almost fanatical yet passionate absorption. His face is noted to be different from others, not quite masculine or childish, and it carries a timeless quality, almost resembling a blend of masculine and feminine elements. The narrator feels that Demian is unimaginarily different, almost like an animal, a spirit, or an image. Overall, his appearance evokes a sense of attraction and repulsion, and he is portrayed as having a unique and striking presence.',
 'source_documents': [Document(id='78ae7f7d-6c61-4070-8719-3f456e47a6a7', metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 53, 'page_label': '54'}, page_content="DEMIAN \nwith 

In [ ]:
# 답변 본문을 마크다운 형태로 출력
from IPython.display import Markdown, display

display(Markdown(result["result"]))


Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet, almost fanatical yet passionate absorption. His face is noted to be different from others, not quite masculine or childish, and it carries a timeless quality, almost resembling a blend of masculine and feminine elements. The narrator feels that Demian is unimaginarily different, almost like an animal, a spirit, or an image. Overall, his appearance evokes a sense of attraction and repulsion, and he is portrayed as having a unique and striking presence.

In [ ]:
# 어떤 원문 chunk가 근거로 사용되었는지 확인
for i, source_doc in enumerate(result["source_documents"], start=1):
    print(f"[source {i}]")
    print(source_doc.page_content[:500])
    print()


[source 1]
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not 

[source 2]
DEMI.\_N 
intense fervour. Then my dream returned at once-our 
gateway and the coat-of-arms, my mother a_!ld the strange 
woman whose features I saw with such a preternatural 
darity that I was able to draw her portrait from memory 
the same evening . . 
When this drawing was completed a few days later 
and had been painted in almost unconsciously during 
the odd ten minutes in betweentimes, I hung it on my 
wall the same evening. pushed the study lamp in front 
of it and

### RAG를 사용하지 않은 LLM 호출도 해보기

같은 질문을 문서 검색 없이 바로 LLM에 물어보면, 근거 기반 답변과 어떤 차이가 나는지 비교 가능


In [ ]:
# 검색 없이 일반 LLM 호출만 수행
llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


"Demian" is a novel by Hermann Hesse, published in 1919. The story follows a young man named Emil Sinclair as he navigates his journey of self-discovery and explores themes of duality, identity, and the search for meaning. 

In terms of physical appearance, the character Demian, who plays a significant role in Sinclair's life, is often described as having an intense and charismatic presence. He is depicted as confident, with a strong personality that draws others to him. However, specific details about his looks can vary based on individual interpretations and adaptations of the story.

If you are referring to a specific adaptation of "Demian," such as a film or graphic novel, please provide more details, and I can give you a more tailored description!

### Quiz

결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 더 신뢰할 수 있겠다고 느끼셨나요?


### Answer

검색된 source document를 함께 확인할 수 있기 때문에, 답변의 근거를 추적할 수 있습니다.


## 6. 완성 예제


In [ ]:
# 문서 로딩과 벡터 저장에 사용할 주요 모듈을 미리 import
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
# RetrievalQA는 템플릿 흐름 유지를 위해 호환 버전에서 사용
from langchain.chains import RetrievalQA


In [ ]:
# 완성 예제에서도 동일한 토큰 길이 함수를 사용
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)


In [ ]:
# PDF 문서를 페이지 단위 Document로 로드
# 최신 문서 기준으로는 load() 후 splitter를 별도로 적용하는 방식이 더 안정적
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load()


In [ ]:
# Step 2. Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=tiktoken_len
)
texts = text_splitter.split_documents(pages)


In [ ]:
# Step 3. Embedding + VectorStore
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
docsearch = InMemoryVectorStore.from_documents(documents=texts, embedding=embedding_model)


In [ ]:
# Step 4. Retriever + LLM
llm_gpt = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

qa = RetrievalQA.from_chain_type(
    llm_gpt,
    chain_type="stuff",
    retriever=docsearch.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    ),
    return_source_documents=True
)


In [ ]:
# Step 5. Question Answering
query = "how demian looks like"
result = qa.invoke({"query": query})

display(Markdown(result["result"]))


Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet, almost fanatical yet passionate absorption. His face is noted to be different from others, not quite masculine or childish, and it carries a timeless quality, almost resembling a blend of masculine and feminine elements. The narrator feels that Demian is uncommonly different, almost like an animal, a spirit, or an image, making it difficult to categorize him in conventional terms. Overall, he is portrayed as having a unique and striking appearance that sets him apart from others.